In [1]:
import json
import os
import pandas as pd
import pickle

# Data Extraction and Summarization

- subject data are in 'subId.json' files
- read each file and summarie the important bird task trial information and survey responses for further analysis

*** specify the subset data source ***

In [58]:
# specify the folder name containing the subset of data collected
label = "pilot1"


In [59]:
def read_json_files(folder_path):
    """
    Reads all JSON files in a given folder and returns a dictionary
    containing file names as keys and parsed JSON data as values.
    
    Args:
    - folder_path (str): Path to the folder containing JSON files.
    
    Returns:
    - dict: A dictionary containing file names as keys and parsed JSON data as values.
    """
    json_data = {}
    for filename in os.listdir(folder_path):
        if filename.endswith('.json'):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r') as file:
                json_data[filename] = json.load(file)
    return json_data
    

In [60]:
def summarize_save_data(json_data, label):
    """
    Summarizes the structure of JSON data.
    And save the extracted survey and bird task data to csv.
    
    Args:
    - json_data (dict): A dictionary containing JSON data.
    - label (str): folder name containing the subset of data collected. 
    """
    freq_fail_summary = {} # Summary dictionary containing failed count for each survey
    freq_fail_answer = {} # Summary dictionary containing failed answer for each survey
    failed_participants = {} # Summary dictionary containing participants failed more than 3 attention questions with counts
    dfs_trials = pd.DataFrame() # Dataframes containing bird task trials
    dfs_surveys = pd.DataFrame() # Dataframes containing survey responses
    dfs_rpms = pd.DataFrame() # Dataframes containing rpm responses
    saved_participants = [] # Summary of saved participant data

    # iterate over all the participants .json files
    for filename, data in json_data.items():
        # summarize index dictionary for important trials
        print(f"Summary for file: {filename}")
        data_summary = _summarize(data)
        # summarize frequently failed attention questions
        for n in data_summary['freq_fail'].keys():
            if n not in freq_fail_summary.keys():
                    freq_fail_summary[n] = 1
                    freq_fail_answer[n] = [data_summary['freq_fail'][n]]
            else: 
                freq_fail_summary[n] += 1
                freq_fail_answer[n].append(data_summary['freq_fail'][n])
        # excluse participants who failed more than three attention questions
        if data_summary['count'] > 3:
            print("---Failed more than THREE attention questions!---")
            failed_participants[filename] = data_summary['count']
        
        # for all subjects extract and save important data 
        trials_dict, surveys_dict, rpms_dict = extract_data(data, data_summary) 
        subject_id = filename.rsplit('.', 1)[0]
        df_trials = pd.DataFrame(trials_dict, index = [subject_id]*200)
        df_surveys = pd.DataFrame(surveys_dict, index = [subject_id])
        # add data_summary['count'] to df_surveys
        df_surveys['total_failed_attention_checks'] = data_summary['count']
        df_rpms = pd.DataFrame(rpms_dict, index = [subject_id])
        saved_participants.append(subject_id)

        dfs_trials = pd.concat([dfs_trials, df_trials], axis = 0)
        dfs_surveys = pd.concat([dfs_surveys, df_surveys], axis = 0)
        dfs_rpms = pd.concat([dfs_rpms, df_rpms], axis = 0)

    dfs_trials = dfs_trials.rename_axis('subId')
    dfs_surveys = dfs_surveys.rename_axis('subId')
    dfs_rpms = dfs_rpms.rename_axis('subId')
    # save trials data and survey data to csv
    output_file = 'processed_all/' + label + "_trials.csv"
    dfs_trials.to_csv(output_file)
    output_file = 'processed_all/' + label + "_surveys.csv"
    dfs_surveys.to_csv(output_file)
    output_file = 'processed_all/' + label + "_rpms.csv"
    dfs_rpms.to_csv(output_file)
    print(f"---FILE SAVED ({len(saved_participants)})---") # report the number of participants saved
    
    # save failed participants
    print(failed_participants) 
    output_pkl = 'processed_all/' + label + '_failed_participants.pkl'
    with open(output_pkl, 'wb') as file:
        pickle.dump(failed_participants, file)
    
    # summarize frequent failed attention questions
    output_pkl = 'processed_all/' + label + '_failed_qs.pkl'
    with open(output_pkl, 'wb') as file:
        pickle.dump(freq_fail_answer, file)
    # save list of participant ID
    output_pkl = 'processed_all/' + label + '_subID.pkl'
    with open(output_pkl, 'wb') as file:
        pickle.dump(saved_participants, file)
    
    # print summary of failed attention questions
    print(freq_fail_summary)
    print(freq_fail_answer)   

In [61]:
def _summarize(data):
    """
    Helper function to summarize JSON data structure.
    
    Args:
    - data: JSON data (dict) to be summarized.
    
    Returns:
    - dict: A dictionary containing labeled task index.
    """
    print(f"{' '*(4)}Data length: ", len(data))
    
    view_history = [] # view_history
    view_history_idx = []
    comprehensions = [] # responses & num_errors
    comprehensions_idx = []
    surveys = [] # responses & survey
    surveys_idx = []
    trials = [] # bird_position (blocks)
    trials_idx = []
    stimulus = [] # stimulus (rpm)
    stimulus_idx = []
    others = [] # others
    others_idx = []

    # use dict keys to identify trials
    for item in range(len(data)):
        # 9 before trials (4 more if failed comprehension) + 10 after trials (more if failed attention check) = 19 total
        if 'view_history' in data[item].keys():
            view_history.append(data[item])
            view_history_idx.append(item)
        # 2 comprehensions (3 if failed once)
        elif 'responses' and 'num_errors' in data[item].keys(): 
            comprehensions.append(data[item])
            comprehensions_idx.append(item)
        # 23 + 200 (50 trials * 4 blocks) = 223 
        elif 'bird_position' in data[item].keys():
            trials.append(data[item])
            trials_idx.append(item)
        # 16 surveys (2 in parts) + demographics = 19 total
        elif 'responses' and 'survey' in data[item].keys(): 
            surveys.append(data[item])
            surveys_idx.append(item)
#             for i in surveys:
#                 print(i['survey']) 
        # 26 image stimulus responses (18 rpm)
        elif 'stimulus' in data[item].keys(): 
            stimulus.append(data[item])
            stimulus_idx.append(item)
        # screen checks # 10 screen checks
        else: 
            others.append(data[item])
            others_idx.append(item)
            
    # detailed labels (rpm and blocks)
    rpm = stimulus[8:]
    rpm_idx = stimulus_idx[8:]
    # the last 200 trials contains 4*50 bird task trial data
    blocks = trials[-200:]
    blocks_idx = trials_idx[-200:]
    if len(trials_idx) != 223:
        print(f"***This is a warning for bird task trial inaccuracy ({len(trials_idx)})***")
               
    # check number of failed attention questions
    freq_fail, count = attention_check(surveys)
    
    # output summary data index
    data_summary = {'comprehensions_idx': comprehensions_idx,
                    'blocks_idx': blocks_idx,
                    'surveys_idx': surveys_idx,
                    'rpm_idx': rpm_idx,
                    'freq_fail': freq_fail,
                    'count': count
                   } 
    
    return data_summary

#     print(len(view_history_idx))
#     print(len(comprehensions_idx))
#     print(len(trials_idx))
#     print(len(surveys_idx))
#     print(len(stimulus_idx))
#     print(len(rpm_idx))
#     print(len(blocks_idx))
#     print(len(others_idx))     
#     total = len(view_history_idx) + + len(comprehensions_idx) + len(trials_idx) + len(surveys_idx) + len(stimulus_idx) + len(others_idx)
#     print(total)

In [62]:
def extract_data(data, data_summary):
    """
    Extract needed data.
    params extracted: 'trials': ['bird_position', 'bag_position', 'bucket_position', 'completed', 'stayed', 'block', 'randomized', 'trial', 'time_elapsed'],
                      'surveys': ['survey', 'responses', 'item_order', 'time_elapsed'],
                      'rpm': ['stimulus', 'choice_order', 'correct', 'choice', 'accuracy']
    
    Args:
    - data: JSON data (dict) to be summarized.
    - data_summary: dict containing summarized index with labels
    
    Returns:
    - dict: dictionary containing extracted data with labels.
    """
    # Extract important bird task trial data
    keys_to_extract = ['trial', 'block', 'randomized', 
                       'bird_position', 'bag_position', 'bucket_position', 
                       'completed', 'stayed', 'time_elapsed']
    blocks_idx = data_summary['blocks_idx']
    trials_dict = []
    for t in blocks_idx:
        trial = data[t]
        # Extract dictionary using dictionary comprehension
        extracted_dict = {key: trial[key] for key in keys_to_extract}
        trials_dict.append(extracted_dict)
#     # quality check 
#     a = 0
#     b = 0
#     c = 0
#     d = 0
#     for t in trials_dict:
#         if t['block'] == 1:
#             a+=1
#         if t['block'] == 2:
#             b+=1
#         if t['block'] == 3:
#             c+=1    
#         if t['block'] == 4:
#             d+=1 
#     print(a,b,c,d)
            
    # Extract important survey data
    surveys_idx = data_summary['surveys_idx']
    surveys_dict = {}
    for s in surveys_idx:
        survey = data[s]
        _name = survey['survey'] 
        if 'responses' in survey.keys():
            _responses = survey['responses']
            # identify demographic survey
            if _name == 'demographics':
                for i in _responses.keys():
                        surveys_dict[_name+'_'+i] = _responses[i]
            # other surveys
            else:
                sorted_reponses = dict(sorted(_responses.items()))
                # if there is infrequency questions, label last question as _infrequency
                if 'infrequency' in survey.keys():
                    for i in range(len(sorted_reponses.keys())-1):
                        q = list(sorted_reponses.keys())[i]
                        surveys_dict[_name+'_'+q] = sorted_reponses[q]
                    q = list(sorted_reponses.keys())[-1]
                    surveys_dict[_name+'_'+'infrequency'] = sorted_reponses[q]
                else:
                    for i in sorted_reponses.keys():
                        surveys_dict[_name+'_'+i] = sorted_reponses[i]

            surveys_dict[_name+'_time_elapsed'] = survey['time_elapsed']
        else:
            print(f"****************{_name}: no responses****************")
        
    # Extract important rpm data
    rpm_keys = ['choice_order', 'correct', 'choice', 'accuracy']
    rpm_idx = data_summary['rpm_idx']
    rpms_dict = {}
    count=0
    acc=0
    for r in rpm_idx:
        rpm = data[r]
        if 'accuracy' in rpm.keys():
            # Extract dictionary using dictionary comprehension
            _stimulus = rpm['stimulus'].split('/')[-1].split('.')[0]
            for k in rpm_keys:
                rpms_dict[_stimulus+'_'+k] = str(rpm[k])
            
            count+=1
            if rpm['accuracy'] == 1:
                acc+=1
    print(f'rpm accuracy: {acc}/{count}')

    return trials_dict, surveys_dict, rpms_dict

In [63]:
def attention_check(surveys): 
    """
    Helper function to report attention check question failed counts.
    
    Args:
    - surveys: dict containing survey data for each participant.
    
    Returns:
    - dict: A dictionary containing labeled frequent failed attention questions.
    - count: integer count of total failed attention question for each participant.
    """
    count = 0
    freq_fail = {}
    for survey in surveys:
        n = survey['survey']
        if 'infrequency' in survey.keys(): 
            if survey['infrequency'] != 0:
#                 print(n, sorted(survey['responses'].items())[-1][1])
                count += 1
                freq_fail[n] = sorted(survey['responses'].items())[-1][1]
                
    print(f"{' '*(4)}total failed questions: ", count)
    return freq_fail, count

In [64]:
# Example usage
folder_path = "/Users/Kary/Project/bird_data/"
os.chdir(folder_path)

json_data = read_json_files(folder_path + 'data/' + label)
summarize_save_data(json_data, label)

Summary for file: ycbdn8pfts8t6o5ap5ga7k6k.json
    Data length:  297
    total failed questions:  0
rpm accuracy: 6/9
Summary for file: 6zi2jx295mw2szvlh65ta6g8.json
    Data length:  297
    total failed questions:  0
rpm accuracy: 5/9
Summary for file: f26s84vrsse3nk62dn1gjyuu.json
    Data length:  299
    total failed questions:  0
rpm accuracy: 6/9
Summary for file: kgug5wdya6nimsjwf2q38997.json
    Data length:  300
    total failed questions:  1
rpm accuracy: 4/9
Summary for file: ekcufb4c6iv3fgyil9zwxosy.json
    Data length:  297
    total failed questions:  0
rpm accuracy: 7/9
Summary for file: w5h0qgfhzwuz1jsyyzt7hf7e.json
    Data length:  304
    total failed questions:  1
rpm accuracy: 7/9
Summary for file: 9gq71ekt7kbzzaz452okx6tb.json
    Data length:  298
    total failed questions:  0
rpm accuracy: 3/9
Summary for file: bt4f7rulyep55pvypy4txa7r.json
    Data length:  308
***This is a warning for bird task trial inaccuracy (224)***
    total failed questions:  1
rpm a